# 02 — Leer documentos y construir chunks

**Paso 4.** Documentos → metadatos → secciones → fragmentos inspeccionables.

Este notebook trabaja localmente, sin API key, llamadas al LLM ni embeddings. Solo lee `corpus/*.md`; no incorpora la rúbrica de evaluación. Ejecutar de arriba abajo con **Python (rag-demo)**.

Elegimos secciones completas porque estos documentos son breves. No pretendemos alcanzar un número fijo de tokens: primero preservamos el sentido de reglas y excepciones.

## Cómo trabajar con este material

Un notebook combina **celdas Markdown**, que explican el razonamiento, con **celdas de código**, que ejecutan Python. Seleccioná una celda y presioná Shift+Enter para ejecutarla y avanzar. El kernel es el proceso de Python que mantiene las variables en memoria: cerrar una celda no borra esas variables.

Ejecutá de arriba hacia abajo. Si reiniciás el kernel, volvé a ejecutar desde el comienzo. Editar una celda no modifica automáticamente las variables: hay que ejecutarla de nuevo. Esto importa especialmente al cambiar una pregunta o una configuración.

Cada bloque tiene una explicación previa y una guía para interpretar su salida. Conviene hacer una pausa antes de ejecutar: **¿qué entra, qué transformación ocurre y qué debería salir?** No es necesario memorizar la sintaxis; sí poder relacionar cada operación con su propósito.


### Localizar los archivos sin depender de dónde se abrió el notebook

El primer bloque prepara herramientas y determina qué archivos leerá. Las importaciones no ejecutan por sí mismas el procesamiento del corpus.

| Herramienta | Uso en este notebook |
|---|---|
| `Path` | Encontrar carpetas, listar archivos, leerlos y escribir resultados. |
| `Counter` | Contar cuántos chunks tiene cada documento. |
| `date` | Interpretar fechas y comparar el inicio y el fin de una vigencia. |
| `hashlib` | Calcular una huella de los bytes de cada fuente. |
| `json` y `yaml` | Convertir entre texto en esos formatos y estructuras de Python. |
| `re` | Reconocer delimitadores y encabezados mediante patrones de texto. |
| `display`, `Markdown` | Mostrar títulos legibles dentro de Jupyter. |

`Path.cwd()` devuelve la carpeta de trabajo del kernel. Puede variar según cómo abrimos Jupyter. Recorremos esa carpeta y sus antecesoras con `next(...)` hasta encontrar `requirements.in` y `corpus/`. Así evitamos escribir una ruta absoluta que solo funcionaría en esta computadora. El valor de respaldo `None` permite detectar si no se encontró el proyecto.

`glob("*.md")` selecciona archivos Markdown directamente dentro de `corpus/`; no recorre otras carpetas. `sorted(...)` fija su orden para que las ejecuciones sean comparables. Excluimos así `.env`, notebooks y rúbricas por construcción, en vez de leer todo el proyecto y luego intentar eliminar lo que sobra.

`if not archivos` detecta una lista vacía. `len(archivos)` cuenta sus elementos y el bucle `for` imprime sus nombres. Todavía no leímos los contenidos.


In [ ]:
from pathlib import Path
from collections import Counter
from datetime import date
import hashlib
import json
import re
import yaml
from IPython.display import display, Markdown

actual = Path.cwd().resolve()
raiz = next((p for p in [actual, *actual.parents]
             if (p / "requirements.in").is_file() and (p / "corpus").is_dir()), None)
if raiz is None:
    raise RuntimeError("Abrí este notebook dentro de rag-demo.")
archivos = sorted((raiz / "corpus").glob("*.md"))
if not archivos:
    raise ValueError("El corpus está vacío.")
print(f"Documentos encontrados: {len(archivos)}")
for archivo in archivos:
    print(archivo.name)

**Salida esperada.** Seis nombres de archivo, todos del corpus. Si aparece cero, revisar la ubicación y el formato de los archivos. Tener seis archivos no demuestra que estén bien formados: la siguiente etapa verifica su contenido.

## 1. Separar metadatos del cuerpo

El bloque entre `---` es YAML (*front matter*). Usamos `safe_load` y comprobamos campos obligatorios y fechas. No descartamos documentos por estado: ese filtro pertenece a una etapa posterior.

Este lector está diseñado para el formato de nuestro corpus Markdown, no como parser universal de Markdown.

### Convertir un archivo Markdown en un documento de Python

Definimos `leer_documento(archivo)` para repetir la misma transformación con cada fuente. `def` crea la función; su cuerpo se ejecuta cuando la llamamos. El argumento `archivo` es un `Path`, y el resultado será un diccionario.

**Lectura y separación.** `read_text(encoding="utf-8-sig")` lee el texto y admite una posible marca inicial de UTF-8. El patrón de `re.match` exige un bloque YAML al principio:

- `\A` y `\Z` anclan el patrón al comienzo y al final del texto.
- `---` y los saltos de línea identifican los delimitadores del front matter.
- `(.*?)` captura los metadatos con la menor extensión que permita cerrar el bloque.
- `(.*)` captura el cuerpo restante; `re.S` permite que el punto incluya saltos de línea.

`group(1)` y `group(2)` recuperan esas dos capturas. Este patrón sirve para el formato acordado del corpus; no pretende interpretar cualquier documento Markdown.

**Validación.** `yaml.safe_load` convierte YAML en estructuras de datos sin usar la carga general de objetos Python. Eso no basta para validar el esquema: comprobamos además que el resultado sea un diccionario y tenga los campos esperados. `CAMPOS` es un conjunto; `CAMPOS - metadatos.keys()` produce los nombres que faltan. `or` permite detener la evaluación si el objeto ni siquiera es un diccionario.

Para los campos obligatorios comprobamos tipo `str` y contenido no vacío con `strip()`. Las fechas están entre comillas en las fuentes: llegan como texto y `date.fromisoformat` las interpreta como fechas. `vigente_hasta` puede ser `None`, que representa el `null` de YAML. Cuando hay fecha de fin, no puede ser anterior a la de inicio. Esta comprobación valida el intervalo; no decide por sí sola si la política es aplicable a una consulta.

**Resultado.** El diccionario guarda `metadata`, `body`, una ruta relativa en `source` y una huella `source_sha256`. `relative_to(raiz).as_posix()` usa una referencia portable con barras `/`. La huella se calcula sobre los bytes originales; incluso un cambio de saltos de línea puede alterarla. Sirve para detectar cambios, no para certificar que una política sea verdadera.

La expresión `[leer_documento(archivo) for archivo in archivos]` es una comprensión de listas: llama una vez a la función por archivo. Los dos accesos de `d["metadata"]["document_id"]` recorren diccionarios anidados. Convertir los IDs en un `set` elimina repeticiones; comparar tamaños detecta duplicados. Finalmente, `json.dumps(..., indent=2)` produce una representación legible para inspección: no escribe un archivo.


In [ ]:
CAMPOS = {
    "document_id", "titulo", "version", "estado", "vigente_desde",
    "vigente_hasta", "jurisdiccion", "audiencia", "autoridad", "responsable",
}

def leer_documento(archivo):
    original = archivo.read_text(encoding="utf-8-sig")
    partes = re.match(r"\A---\n(.*?)\n---(?:\n|$)(.*)\Z", original, re.S)
    if not partes:
        raise ValueError(f"Front matter inválido: {archivo.name}")
    metadatos = yaml.safe_load(partes.group(1))
    if not isinstance(metadatos, dict) or CAMPOS - metadatos.keys():
        raise ValueError(f"Metadatos incompletos: {archivo.name}")
    for campo in CAMPOS - {"vigente_hasta"}:
        if not isinstance(metadatos[campo], str) or not metadatos[campo].strip():
            raise ValueError(f"{campo} debe ser texto no vacío: {archivo.name}")
    inicio = date.fromisoformat(metadatos["vigente_desde"])
    fin = metadatos["vigente_hasta"]
    if fin is not None and date.fromisoformat(fin) < inicio:
        raise ValueError(f"Intervalo de vigencia inválido: {archivo.name}")
    if metadatos["estado"] not in {"vigente", "expirada"}:
        raise ValueError(f"Estado desconocido: {archivo.name}")
    cuerpo = partes.group(2).strip()
    if not cuerpo:
        raise ValueError(f"Documento sin cuerpo: {archivo.name}")
    return {
        "metadata": metadatos,
        "source": archivo.relative_to(raiz).as_posix(),
        "source_sha256": hashlib.sha256(archivo.read_bytes()).hexdigest(),
        "body": cuerpo,
    }

documentos = [leer_documento(archivo) for archivo in archivos]
ids = [d["metadata"]["document_id"] for d in documentos]
assert len(ids) == len(set(ids)), "Hay document_id duplicados."
for doc in documentos:
    m = doc["metadata"]
    print(m["document_id"], "|", m["estado"], "|", m["jurisdiccion"], "|", m["autoridad"])
print("\nEjemplo de metadatos:")
print(json.dumps(documentos[0]["metadata"], ensure_ascii=False, indent=2))

**Qué mirar.** Deben aparecer seis documentos con estado, jurisdicción y autoridad. La política expirada debe seguir presente. No corregimos la FAQ ni resolvemos contradicciones durante la lectura: preservar fielmente las fuentes permite estudiar esos problemas después.

## 2. Dividir por secciones

Cada encabezado `##` inicia una unidad. Conservamos el texto previo como introducción compartida: allí están, por ejemplo, el aviso de archivo histórico y el límite de autoridad de la FAQ.

La introducción y los metadatos se repetirán en cada fragmento. Este solapamiento contextual consume espacio, pero evita perder advertencias al recuperar una sección aislada.

### Encontrar límites de secciones y conservar la introducción

Un chunk es la unidad que podremos recuperar. Cortar cada cierto número de caracteres es sencillo, pero puede separar una condición de su excepción. Para este corpus usamos la estructura que ya escribió el autor: cada encabezado `##` inicia una sección.

`re.finditer(r"^## (.+)$", cuerpo, re.M)` devuelve coincidencias con sus posiciones. Con `re.M`, `^` y `$` se aplican a cada línea. No buscamos `#` porque ese es el título del documento. Tampoco usamos este método como parser de Markdown con bloques de código: nuestro corpus no tiene encabezados aparentes dentro de ellos.

`list(...)` permite contar e indexar esas coincidencias. `cuerpo[:encabezados[0].start()]` toma todo lo anterior a la primera sección: es la introducción compartida. La notación `texto[inicio:fin]` extrae una porción incluyendo el comienzo y excluyendo el final.

`enumerate(encabezados)` entrega tanto la posición `i` como el encabezado. La sección termina donde empieza el siguiente; para la última usamos `len(cuerpo)`. `encabezado.end()` permite comenzar después de su título, evitando incluirlo dos veces.

Un segundo patrón extrae IDs como `V1` o `A2`: letras mayúsculas, números y un punto seguido de espacio. Guardamos por separado ID, título y texto. `append(...)` agrega cada sección a la lista. Un conjunto de IDs comprueba que no haya duplicados dentro del documento.

La función devuelve dos valores; la asignación a `doc["introduction"], doc["sections"]` los distribuye entre dos campos nuevos. Agregamos información al documento sin reemplazar su cuerpo original, que luego servirá para comprobar integridad.


In [ ]:
def dividir_secciones(cuerpo):
    encabezados = list(re.finditer(r"^## (.+)$", cuerpo, re.M))
    if not encabezados:
        raise ValueError("El documento debe tener secciones con encabezados ##.")
    introduccion = cuerpo[:encabezados[0].start()].strip()
    secciones = []
    for i, encabezado in enumerate(encabezados):
        fin = encabezados[i + 1].start() if i + 1 < len(encabezados) else len(cuerpo)
        titulo = encabezado.group(1).strip()
        coincidencia = re.match(r"([A-Z]+[0-9]+)\. ", titulo)
        if not coincidencia:
            raise ValueError(f"Sección sin identificador: {titulo}")
        texto = cuerpo[encabezado.end():fin].strip()
        if not texto:
            raise ValueError(f"Sección vacía: {titulo}")
        secciones.append({"section_id": coincidencia.group(1), "title": titulo, "text": texto})
    if len({s["section_id"] for s in secciones}) != len(secciones):
        raise ValueError("Hay identificadores de sección duplicados.")
    return introduccion, secciones

for doc in documentos:
    doc["introduction"], doc["sections"] = dividir_secciones(doc["body"])
    print(doc["metadata"]["document_id"], "→", [s["section_id"] for s in doc["sections"]])

**Salida esperada.** Cada documento aparece acompañado por sus IDs de sección. A1 y A2 pertenecen al anexo; los demás documentos tienen tres secciones cada uno. El total es 17. La introducción no se convierte en un chunk independiente: se incorpora como contexto en los chunks de ese documento.

## 3. Construir el texto de cada chunk

`text` es la sección original. `context_sections` conserva otras secciones necesarias y su procedencia. `retrieval_text` combina metadatos, introducción, contexto y sección: será el texto candidato para la siguiente etapa.

**Decisión explícita para este corpus:** A2 dice «el personal comprendido». Adjuntamos A1 para conservar el país de contratación, la vigencia y el plazo. La relación A2 → A1 está declarada abajo, no inferida mágicamente por el splitter. En corpus mayores habría que diseñar otras estrategias.

Esto no convierte cada chunk en evidencia suficiente para toda pregunta: puede seguir siendo necesario recuperar varios fragmentos, como la regla global y la de precedencia.

### Diseñar la estructura de datos que usará la recuperación

Esta función recorre documentos y, dentro de cada uno, sus secciones. Los dos bucles reflejan esa jerarquía. Una sección principal produce un chunk, aunque le adjuntemos contexto adicional.

`DEPENDENCIAS` usa una **tupla** `(document_id, section_id)` como clave. Así A2 de un documento no se confunde con una sección de otro. `.get(clave, [])` devuelve las dependencias declaradas o una lista vacía. El diccionario `secciones` permite encontrar A1 directamente por su ID. Si una dependencia declarada no existe, el acceso falla: es mejor corregir la referencia que omitirla en silencio.

`.copy()` crea copias superficiales de los diccionarios para no compartir el mismo contenedor al armar los chunks. Es suficiente aquí porque no modificamos estructuras anidadas de esas copias.

`yaml.safe_dump` transforma metadatos en texto; `allow_unicode=True` mantiene caracteres como tildes y `sort_keys=False` conserva el orden de inserción. La lista `partes` reúne cabecera, introducción, contexto adicional y sección. `"\n\n".join(...)` las une con separaciones legibles, omitiendo cadenas vacías.

| Campo del chunk | Por qué lo conservamos |
|---|---|
| `chunk_id` | Identifica la sección dentro de su documento, por ejemplo `VIAJES-AR-V2::A2`. |
| `document_id`, `source`, `source_sha256` | Permiten volver a la fuente y detectar si cambió. |
| `metadata` | Conserva versión, audiencia, autoridad y vigencia como datos accesibles sin analizar otra vez el texto. |
| `section_id`, `section_title`, `text` | Identifican y preservan la sección principal original. |
| `introduction`, `context_sections` | Explicitan el contexto que agregamos y de dónde proviene. |
| `retrieval_text` | Es el texto compuesto que proponemos usar al calcular representaciones para búsqueda. |
| `characters`, `words` | Dan una medida simple del tamaño del texto compuesto. No cuentan tokens del modelo. |

`len(texto.split())` cuenta fragmentos separados por espacios en blanco. `Counter(...)` resume los chunks por documento, y `min`/`max` muestran los extremos de tamaño.

Agregar metadatos al texto hace visible su contexto, pero no garantiza que la búsqueda respete la autoridad: sigue haciendo falta una política de selección y uso de evidencia. Repetir A1 también puede aumentar la similitud entre chunks del anexo. Esa es una decisión que evaluaremos al observar resultados reales.


In [ ]:
DEPENDENCIAS = {("VIAJES-AR-V2", "A2"): ["A1"]}

def construir_chunks(documentos):
    resultado = []
    for doc in documentos:
        m = doc["metadata"]
        secciones = {s["section_id"]: s for s in doc["sections"]}
        for seccion in doc["sections"]:
            contexto = [secciones[sid].copy() for sid in
                        DEPENDENCIAS.get((m["document_id"], seccion["section_id"]), [])]
            cabecera = yaml.safe_dump(m, allow_unicode=True, sort_keys=False).strip()
            partes = ["Metadatos de la fuente:\n" + cabecera, doc["introduction"]]
            for extra in contexto:
                partes.append("Contexto de la misma fuente — " + extra["title"] + "\n" + extra["text"])
            partes.append("## " + seccion["title"] + "\n" + seccion["text"])
            texto_recuperable = "\n\n".join(p for p in partes if p)
            resultado.append({
                "chunk_id": m["document_id"] + "::" + seccion["section_id"],
                "document_id": m["document_id"],
                "source": doc["source"],
                "source_sha256": doc["source_sha256"],
                "metadata": m.copy(),
                "section_id": seccion["section_id"],
                "section_title": seccion["title"],
                "text": seccion["text"],
                "introduction": doc["introduction"],
                "context_sections": contexto,
                "retrieval_text": texto_recuperable,
                "characters": len(texto_recuperable),
                "words": len(texto_recuperable.split()),
            })
    return resultado

chunks = construir_chunks(documentos)
print(f"{len(documentos)} documentos → {len(chunks)} chunks")
for documento, cantidad in Counter(c["document_id"] for c in chunks).items():
    print(documento, cantidad)
print("Tamaños en palabras (no tokens):", min(c["words"] for c in chunks), "a", max(c["words"] for c in chunks))

**Qué debería resultar.** Se mantienen 17 chunks, no 18: A1 sigue teniendo su chunk propio y además acompaña a A2. La duplicación es intencional y está identificada en `context_sections`. Inspeccionar el conteo y los tamaños ayuda a detectar una división inesperada antes de calcular embeddings.

## 4. Inspeccionar fragmentos reales

Mirar A2: ¿incluye todas las condiciones de la excepción? Mirar F2: ¿el plazo incorrecto conserva el aviso de menor autoridad? Los errores deliberados del corpus se mantienen.

### Mirar el texto que recuperaría el sistema

`por_id = {c["chunk_id"]: c for c in chunks}` construye un índice en memoria: un diccionario que asocia cada ID a su chunk. No es un índice vectorial y no hace búsqueda semántica; solo permite acceder por una clave exacta.

Elegimos tres ejemplos por su interés pedagógico: una regla global, una excepción local y una FAQ contradictoria. `display(Markdown(...))` dibuja un título en Jupyter; `print` muestra el contenido literal. Así se puede distinguir la presentación del notebook de los datos que procesará el sistema.

El bucle imprime `retrieval_text`, no solamente `text`, para comprobar el contexto que efectivamente acompañaría a la sección. La multiplicación `"─" * 60` solo crea un separador visual; no forma parte del chunk.


In [ ]:
por_id = {c["chunk_id"]: c for c in chunks}
for chunk_id in ["VIAJES-GLOBAL-V4::V1", "VIAJES-AR-V2::A2", "FAQ-VIAJES-V1::F2"]:
    chunk = por_id[chunk_id]
    display(Markdown("### " + chunk_id))
    print("Fuente:", chunk["source"])
    print(chunk["retrieval_text"])
    print("\n" + "─" * 60)

**Lectura guiada.** En A2 buscá quién está comprendido, dónde ocurrió el taxi, el límite y los requisitos. En F2 comprobá que se vean tanto el plazo de 45 días como la advertencia de menor autoridad. Un fragmento puede estar bien construido y contener una afirmación que no debe usarse como regla vigente: integridad y autoridad son preguntas distintas.

## 5. Comprobar integridad

Verificamos que ninguna sección desaparezca, que no cambie su texto y que se mantengan metadatos y condiciones. Estas comprobaciones validan la preparación; aún no miden calidad de retrieval.

### Convertir las decisiones de diseño en comprobaciones

`assert condición, mensaje` interrumpe la ejecución cuando una condición no se cumple. Lo usamos como comprobación didáctica dentro del notebook; no es un mecanismo para limpiar datos ni reemplaza validaciones de producción. Python puede omitir los `assert` al ejecutarse con optimizaciones.

Primero comprobamos cantidad e identidad: no debe haber dos chunks con el mismo ID y debe existir uno por sección. Después reconstruimos el cuerpo usando la introducción, los títulos y los textos guardados. La igualdad exacta con `body` confirma que, para este formato, el splitter no perdió ni cambió contenido. Comparamos contra el cuerpo ya separado del front matter y recortado con `strip()`, no contra todos los bytes originales del archivo.

Los siguientes chequeos comparan texto y metadatos de cada chunk con su fuente. `in` verifica que la sección principal esté incluida en el texto compuesto. `any(...)` confirma que al menos un fragmento siga marcado como expirado.

Por último se comprueban decisiones deliberadas de la demo: conservar el 45 de la FAQ y su advertencia, y adjuntar A1 a A2 con el alcance argentino y el monto de la excepción. Estos chequeos dependen de este corpus; si cambiamos las políticas intencionalmente, debemos revisar también qué esperamos de ellas.


In [ ]:
assert len(por_id) == len(chunks), "chunk_id duplicados."
assert len(chunks) == sum(len(d["sections"]) for d in documentos)
for doc in documentos:
    reconstruido = "\n\n".join([doc["introduction"]] + [
        "## " + s["title"] + "\n\n" + s["text"] for s in doc["sections"]
    ])
    assert reconstruido == doc["body"], "Se perdió o modificó texto al dividir."
    for seccion in doc["sections"]:
        c = por_id[doc["metadata"]["document_id"] + "::" + seccion["section_id"]]
        assert c["text"] == seccion["text"]
        assert c["metadata"] == doc["metadata"]
        assert c["text"] in c["retrieval_text"]

assert any(c["metadata"]["estado"] == "expirada" for c in chunks)
assert "45 días" in por_id["FAQ-VIAJES-V1::F2"]["text"]
assert "No reemplaza políticas" in por_id["FAQ-VIAJES-V1::F2"]["retrieval_text"]
excepcion = por_id["VIAJES-AR-V2::A2"]
assert [s["section_id"] for s in excepcion["context_sections"]] == ["A1"]
assert "personal contratado por Andes Retail Argentina" in excepcion["retrieval_text"]
assert "ARS 18.000" in excepcion["text"]
print("Integridad verificada: texto, metadatos, excepción y distractores conservados.")

**Interpretación.** Si aparece el mensaje de integridad, estas condiciones se cumplieron. Eso no demuestra todavía que una pregunta recupere los chunks correctos, que la estrategia sea óptima ni que un LLM use bien la evidencia. Si falla una comprobación, detenerse y revisar la causa antes de exportar.

## 6. Guardar para la siguiente etapa

Generamos `data/chunks.jsonl`: un objeto JSON por línea. El archivo se puede volver a generar al cambiar el corpus; no editarlo manualmente. Los identificadores se mantienen si no cambia la sección y la huella SHA-256 permite detectar cambios en la fuente.

El manifiesto documenta la estrategia. No es un índice vectorial: todavía no calculamos embeddings ni recuperamos resultados.

### Persistir resultados para no depender de la memoria del kernel

Hasta ahora `chunks` existe solo en memoria. Al reiniciar el kernel desaparecería. Guardarlo permite inspeccionarlo con otras herramientas y usarlo en el siguiente notebook sin copiar código de preparación.

`mkdir(exist_ok=True)` crea `data/` si hace falta y permite reutilizarla. `json.dumps(c, ensure_ascii=False)` convierte cada diccionario en una línea JSON con tildes legibles. Al agregar `"\n"` y unir las líneas obtenemos **JSONL**: cada línea es un registro independiente. No es un único arreglo JSON que se lee con una sola llamada a `json.load`.

`write_text` escribe el resultado en UTF-8 y reemplaza el archivo anterior del mismo nombre. Es intencional: se trata de datos derivados que regeneramos a partir del corpus. No modifica las fuentes originales.

El manifiesto es un JSON normal, formateado con `indent=2`. Registra versión del formato, estrategia, dependencia A2 → A1, conteos y huellas de las fuentes. La versión del formato describe la estructura de nuestros datos, no la versión de una política.

Finalmente leemos el JSONL con `splitlines()` y aplicamos `json.loads` a cada línea. La igualdad `recargados == chunks` comprueba una ida y vuelta completa: lo que recuperamos del disco coincide con lo que teníamos en memoria. Esto detecta problemas de serialización antes de que aparezcan en el siguiente notebook.


In [ ]:
salida = raiz / "data"
salida.mkdir(exist_ok=True)
ruta_chunks = salida / "chunks.jsonl"
ruta_chunks.write_text(
    "".join(json.dumps(c, ensure_ascii=False) + "\n" for c in chunks), encoding="utf-8"
)
manifiesto = {
    "schema_version": "1.0",
    "strategy": "markdown_h2_with_introduction_and_explicit_context",
    "dependencies": {"VIAJES-AR-V2::A2": ["VIAJES-AR-V2::A1"]},
    "documents": len(documentos),
    "chunks": len(chunks),
    "sources": [{"source": d["source"], "sha256": d["source_sha256"]} for d in documentos],
}
(salida / "manifest.json").write_text(json.dumps(manifiesto, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
recargados = [json.loads(linea) for linea in ruta_chunks.read_text(encoding="utf-8").splitlines()]
assert recargados == chunks
print("Guardados y releídos:", ruta_chunks.relative_to(raiz), "y data/manifest.json")

**Archivos esperados.** `data/chunks.jsonl` contiene 17 registros y `data/manifest.json` describe cómo se produjeron. No tienen embeddings ni respuestas de evaluación. Si cambia el corpus, volver a ejecutar este notebook antes del siguiente: las huellas registradas permiten comparar fuentes, pero no implementan por sí solas una actualización automática.

## Para discutir en clase

- ¿Qué se perdería al cortar A2 cada 200 caracteres?
- ¿Por qué «vigente» no implica «mayor autoridad»?
- ¿Qué costo tiene repetir contexto en cada chunk? ¿Puede sesgar la similitud?
- ¿Cuándo convendría recuperar el documento padre o fragmentos vecinos?

La estrategia es deliberadamente simple y específica para seis documentos breves. No implementa un límite de tokens ni resuelve automáticamente referencias entre documentos. El paso siguiente será calcular embeddings y mostrar top-k **sin generación**.